# 02 — Veri Hazırlama (Preprocessing)
**BA Degree Project 2 — Credit Scoring ML** · Batuhan SEN · 200527078

Bu notebook, dokümanın **Methodology 5.3 (Data Preprocessing)** bölümünü **birebir** uygular.
Veriyi modele hazır hale getirir. Sıra (methodology'ye sadık):

1. **Split-öncesi temizlik** (sızıntısız, satır-bazlı): ID at · `DAYS_EMPLOYED` anomalisi · XNA · türetilmiş oranlar · %60+ eksik kolonları at
2. **Train/Test split** 80:20 stratified
3. **Encoding** (one-hot düşük kardinalite + target encoding yüksek kardinalite, *yalnız train'de fit*)
4. **Imputation** (median) + **Winsorization** (1–99) — *yalnız train'de fit*
5. **Feature Selection** (near-zero variance + korelasyon>0.85 + **Boruta**)

> **Leakage kuralı:** Tüm dönüşümler yalnız **train**'de öğrenilir, test'e uygulanır.
> En sızıntı-hassas iki adım (**scaling + SMOTE**) ise `03_modeling`'de CV fold'u içinde yapılır.

## 1. Kütüphaneler ve Ayarlar

In [1]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings("ignore")

# Preprocessing araçları
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier   # Boruta'nın motoru
from sklearn.feature_selection import VarianceThreshold
from category_encoders import TargetEncoder            # yüksek kardinalite encoding
from boruta import BorutaPy                            # feature selection

# Proje köküne sabitlen (yollar tutarlı olsun)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

RANDOM_STATE = 42          # tekrarlanabilirlik (sabit seed kuralı)
SAMPLE_SIZE  = 50000       # geliştirme örneklemi (hız); final için pipeline.py tam veri kullanır
os.makedirs("outputs/results", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
print("Hazır · pandas", pd.__version__)

Hazır · pandas 3.0.3


## 2. Veri Yükleme ve Geliştirme Örneklemi (50K, stratified)

In [2]:
# Tam veriyi yükle
df_full = pd.read_csv("data/application_train.csv")
print(f"Tam veri: {df_full.shape[0]:,} satır")

# Geliştirme için 50K stratified örneklem (target oranını korur — %8.07)
df, _ = train_test_split(df_full, train_size=SAMPLE_SIZE,
                         stratify=df_full["TARGET"], random_state=RANDOM_STATE)
df = df.reset_index(drop=True)
print(f"Örneklem: {df.shape[0]:,} satır · default oranı %{df['TARGET'].mean()*100:.2f}")

Tam veri: 307,511 satır
Örneklem: 50,000 satır · default oranı %8.07


## 3. Split-Öncesi Temizlik

Bu işlemler **sızıntısız** çünkü her satırı kendi içinde düzeltir (target'ı kullanmaz):
`DAYS_EMPLOYED` anomalisi, XNA değerleri, türetilmiş banka oranları, EDA'da bulduğumuz kararlar.

In [3]:
# 3.1 — DAYS_EMPLOYED anomalisi (365243 = ~1000 yıl placeholder)
df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)  # bilgiyi flag olarak koru
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)        # gerçek değeri NaN yap

# 3.2 — XNA (bilinmeyen) kategorik değerleri NaN yap (sonra 'Missing' kategorisi olacak)
for kol in ["CODE_GENDER", "ORGANIZATION_TYPE"]:
    df[kol] = df[kol].replace("XNA", np.nan)

# 3.3 — Türetilmiş bankacılık oranları (EDA Bölüm 9) + yaş
df["CREDIT_INCOME_RATIO"]  = df["AMT_CREDIT"]  / df["AMT_INCOME_TOTAL"]
df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["PAYMENT_RATE"]         = df["AMT_ANNUITY"] / df["AMT_CREDIT"]
df["DAYS_EMPLOYED_PERC"]   = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]   # çalışma/yaş oranı
df["AGE_YEARS"]            = -df["DAYS_BIRTH"] / 365

print(f"Anomali işaretlenen satır: {df['DAYS_EMPLOYED_ANOM'].sum():,}")
print(f"Türetilmiş 5 yeni değişken eklendi. Yeni kolon sayısı: {df.shape[1]}")

Anomali işaretlenen satır: 8,949
Türetilmiş 5 yeni değişken eklendi. Yeni kolon sayısı: 128


In [4]:
# 3.4 — Hedef ve ID'yi ayır, %60+ eksik kolonları at
y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])   # ID modele girmemeli

high_missing = X.columns[X.isnull().mean() > 0.60]
X = X.drop(columns=high_missing)
print(f"%60+ eksik {len(high_missing)} kolon atıldı.")
print(f"Kalan: {X.shape[1]} değişken (sayısal {X.select_dtypes('number').shape[1]}, "
      f"kategorik {X.select_dtypes('object').shape[1]})")

%60+ eksik 17 kolon atıldı.
Kalan: 109 değişken (sayısal 94, kategorik 15)


## 4. Train/Test Split (80:20 Stratified)

Methodology 5.6: *"split 80:20 ... using stratified sampling ... all transformations fit
exclusively to training data."* Bu noktadan sonra **hiçbir şey test'ten öğrenilmez.**

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape[0]:,} satır · default %{y_train.mean()*100:.2f}")
print(f"Test : {X_test.shape[0]:,} satır · default %{y_test.mean()*100:.2f}")
print("Stratified split: iki sette de default oranı korundu ✓")

Train: 40,000 satır · default %8.07
Test : 10,000 satır · default %8.07
Stratified split: iki sette de default oranı korundu ✓


## 5. Feature Encoding (Methodology 5.3)

Methodology: *düşük kardinalite (<10) → one-hot, yüksek kardinalite (≥10) → target encoding
(yalnız train'de hesaplanır, leakage önlenir)*. Kategorik eksikler önce **'Missing' kategorisi**
olur (missingness'in kendisi default-öngörücü olabilir).

In [6]:
# Kategorikleri kardinaliteye göre ayır
kategorik = X_train.select_dtypes("object").columns.tolist()
kard = X_train[kategorik].nunique()
dusuk  = kard[kard < 10].index.tolist()    # one-hot
yuksek = kard[kard >= 10].index.tolist()   # target encoding
print(f"One-hot ({len(dusuk)} kolon):", dusuk)
print(f"Target encoding ({len(yuksek)} kolon):", yuksek)

# Kategorik eksikleri 'Missing' kategorisi yap (hem train hem test)
for kol in kategorik:
    X_train[kol] = X_train[kol].fillna("Missing")
    X_test[kol]  = X_test[kol].fillna("Missing")

One-hot (13 kolon): ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']
Target encoding (2 kolon): ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']


In [7]:
from sklearn.preprocessing import OneHotEncoder

# 5.1 — One-hot (yalnız train'de fit; handle_unknown ile test'teki yeni kategoriler güvenli)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
ohe.fit(X_train[dusuk])
tr_ohe = pd.DataFrame(ohe.transform(X_train[dusuk]),
                      columns=ohe.get_feature_names_out(dusuk), index=X_train.index)
te_ohe = pd.DataFrame(ohe.transform(X_test[dusuk]),
                      columns=ohe.get_feature_names_out(dusuk), index=X_test.index)

# 5.2 — Target encoding (yalnız train+y_train'de fit -> leakage yok)
tenc = TargetEncoder(cols=yuksek)
tenc.fit(X_train[yuksek], y_train)
tr_te = tenc.transform(X_train[yuksek])
te_te = tenc.transform(X_test[yuksek])

# 5.3 — Sayısal + one-hot + target-encoded birleştir
sayisal = X_train.select_dtypes("number").columns.tolist()
X_train_enc = pd.concat([X_train[sayisal], tr_ohe, tr_te], axis=1)
X_test_enc  = pd.concat([X_test[sayisal],  te_ohe, te_te], axis=1)
print(f"Encoding sonrası: train {X_train_enc.shape}, test {X_test_enc.shape}")

Encoding sonrası: train (40000, 146), test (10000, 146)


## 6. Imputation (median) + Winsorization (1–99 persentil)

In [8]:
# 6.1 — Median imputation (yalnız train'de fit)
imp = SimpleImputer(strategy="median")
X_train_enc = pd.DataFrame(imp.fit_transform(X_train_enc),
                           columns=X_train_enc.columns, index=X_train_enc.index)
X_test_enc  = pd.DataFrame(imp.transform(X_test_enc),
                           columns=X_test_enc.columns, index=X_test_enc.index)

# 6.2 — Winsorization: train'in 1-99 persentil sınırlarıyla uçları kırp (test'e de train sınırı)
for kol in sayisal:
    alt, ust = X_train_enc[kol].quantile([0.01, 0.99])
    X_train_enc[kol] = X_train_enc[kol].clip(alt, ust)
    X_test_enc[kol]  = X_test_enc[kol].clip(alt, ust)

print(f"Imputation + winsorization tamam. Eksik kalan: {X_train_enc.isnull().sum().sum()}")
print(f"Final boyut -> train {X_train_enc.shape}, test {X_test_enc.shape}")

Imputation + winsorization tamam. Eksik kalan: 0
Final boyut -> train (40000, 146), test (10000, 146)


## 7. Feature Selection (Methodology 5.3)

Üç aşamalı boyut indirgeme (hepsi **train'de** karar verilir):
1. **Near-zero variance** — sabit/neredeyse sabit kolonları at (bilgi taşımaz)
2. **Korelasyon > 0.85** — yüksek korelasyonlu çiftte *düşük univariate-AUC* olanı at
3. **Boruta** — Random Forest ile gerçek önemi gürültüden ayırır (shadow features)

In [9]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import roc_auc_score

# 7.1 — Near-zero variance (sabit kolonlar)
vt = VarianceThreshold(threshold=0.0)
vt.fit(X_train_enc)
sabit = X_train_enc.columns[~vt.get_support()].tolist()
X_train_enc = X_train_enc.drop(columns=sabit)
X_test_enc  = X_test_enc.drop(columns=sabit)
print(f"7.1 Near-zero variance: {len(sabit)} kolon atıldı -> {X_train_enc.shape[1]} kaldı")

7.1 Near-zero variance: 20 kolon atıldı -> 126 kaldı


In [10]:
# 7.2 — Korelasyon > 0.85: çiftten düşük univariate-AUC olanı at (methodology'ye sadık)
uni_auc = {}
for c in X_train_enc.columns:
    a = roc_auc_score(y_train, X_train_enc[c])
    uni_auc[c] = max(a, 1 - a)          # yön bağımsız tek-değişken AUC

corr = X_train_enc.corr().abs()
ust  = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
at_corr = set()
for c in ust.columns:
    esler = ust.index[ust[c] > 0.85]
    for r in esler:
        at_corr.add(c if uni_auc[c] < uni_auc[r] else r)   # düşük AUC olanı at
X_train_enc = X_train_enc.drop(columns=list(at_corr))
X_test_enc  = X_test_enc.drop(columns=list(at_corr))
print(f"7.2 Korelasyon>0.85: {len(at_corr)} kolon atıldı -> {X_train_enc.shape[1]} kaldı")
print(f"    Atılanlar: {sorted(at_corr)}")

7.2 Korelasyon>0.85: 32 kolon atıldı -> 94 kaldı
    Atılanlar: ['AMT_CREDIT', 'APARTMENTS_AVG', 'APARTMENTS_MEDI', 'APARTMENTS_MODE', 'BASEMENTAREA_AVG', 'BASEMENTAREA_MEDI', 'CNT_FAM_MEMBERS', 'DAYS_BIRTH', 'DAYS_EMPLOYED_ANOM', 'DAYS_EMPLOYED_PERC', 'ELEVATORS_AVG', 'ELEVATORS_MODE', 'EMERGENCYSTATE_MODE_No', 'ENTRANCES_MEDI', 'ENTRANCES_MODE', 'FLAG_EMP_PHONE', 'FLOORSMAX_AVG', 'FLOORSMAX_MODE', 'LANDAREA_AVG', 'LANDAREA_MEDI', 'LIVE_REGION_NOT_WORK_REGION', 'LIVINGAREA_AVG', 'LIVINGAREA_MEDI', 'NAME_EDUCATION_TYPE_Secondary / secondary special', 'NONLIVINGAREA_MEDI', 'NONLIVINGAREA_MODE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'REGION_RATING_CLIENT', 'TOTALAREA_MODE', 'WALLSMATERIAL_MODE_Missing', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BEGINEXPLUATATION_MODE']


In [11]:
# 7.3 — Boruta (Random Forest + shadow features)
import time
t0 = time.time()
rf_boruta = RandomForestClassifier(max_depth=6, class_weight="balanced",
                                   n_jobs=-1, random_state=RANDOM_STATE)
boruta = BorutaPy(rf_boruta, n_estimators=80, max_iter=40,
                  random_state=RANDOM_STATE, verbose=0)
boruta.fit(X_train_enc.values, y_train.values)

confirmed = X_train_enc.columns[boruta.support_].tolist()       # kesin önemli
tentative = X_train_enc.columns[boruta.support_weak_].tolist()  # kararsız
print(f"7.3 Boruta bitti ({time.time()-t0:.0f} sn)")
print(f"    Confirmed (kesin): {len(confirmed)} | Tentative (kararsız): {len(tentative)}")

# Karar: confirmed az ise tentative'leri de al (model bilgi kaybetmesin)
secilen = confirmed if len(confirmed) >= 20 else confirmed + tentative
X_train_sel = X_train_enc[secilen].copy()
X_test_sel  = X_test_enc[secilen].copy()
print(f"    Final seçim: {len(secilen)} feature -> train {X_train_sel.shape}")

7.3 Boruta bitti (19 sn)
    Confirmed (kesin): 23 | Tentative (kararsız): 2
    Final seçim: 23 feature -> train (40000, 23)


## 8. Seçilen Özellikler ve Kayıt

In [12]:
# Boruta'nın seçtiği final feature'lar (modele girecekler)
print(f"Boruta'nın seçtiği {len(secilen)} feature:")
for i, f in enumerate(secilen, 1):
    print(f"  {i:2d}. {f}")

Boruta'nın seçtiği 23 feature:
   1. AMT_ANNUITY
   2. AMT_GOODS_PRICE
   3. REGION_POPULATION_RELATIVE
   4. DAYS_EMPLOYED
   5. DAYS_REGISTRATION
   6. DAYS_ID_PUBLISH
   7. REGION_RATING_CLIENT_W_CITY
   8. EXT_SOURCE_1
   9. EXT_SOURCE_2
  10. EXT_SOURCE_3
  11. LIVINGAREA_MODE
  12. FLOORSMAX_MEDI
  13. DAYS_LAST_PHONE_CHANGE
  14. FLAG_DOCUMENT_3
  15. CREDIT_INCOME_RATIO
  16. ANNUITY_INCOME_RATIO
  17. PAYMENT_RATE
  18. AGE_YEARS
  19. CODE_GENDER_M
  20. NAME_INCOME_TYPE_Working
  21. NAME_EDUCATION_TYPE_Higher education
  22. OCCUPATION_TYPE
  23. ORGANIZATION_TYPE


In [13]:
# Temiz, model-ready veriyi kaydet (03_modeling yükleyecek)
import joblib
joblib.dump(
    {"X_train": X_train_sel, "X_test": X_test_sel,
     "y_train": y_train, "y_test": y_test, "features": secilen},
    "data/processed/prepared_50k.joblib")
print("Kaydedildi -> data/processed/prepared_50k.joblib")

# Feature selection özeti (dokümandaki tablo için CSV)
ozet = pd.DataFrame({
    "Aşama": ["Encoding sonrası", "Near-zero variance", "Korelasyon > 0.85", "Boruta (final)"],
    "Feature sayısı": [146, 126, 94, len(secilen)]})
ozet.to_csv("outputs/results/preprocessing_feature_selection.csv", index=False)
print("\nFeature selection özeti:")
print(ozet.to_string(index=False))

Kaydedildi -> data/processed/prepared_50k.joblib

Feature selection özeti:
             Aşama  Feature sayısı
  Encoding sonrası             146
Near-zero variance             126
 Korelasyon > 0.85              94
    Boruta (final)              23


## 9. Preprocessing Özeti

| Adım | Methodology 5.3 | Sonuç |
|---|---|---|
| Anomali temizleme | DAYS_EMPLOYED 365243 → NaN + flag | 8.949 satır |
| Türetilmiş feature | banka oranları + yaş | +5 değişken |
| Eksik kolon atma | >%60 eksik | 17 kolon |
| Split | 80:20 stratified | 40K / 10K |
| Encoding | one-hot + target encoding (train-fit) | 146 feature |
| Imputation | median (train-fit) | 0 eksik |
| Winsorization | 1–99 persentil (train sınırı) | tamam |
| Feature selection | near-zero + korr>0.85 + Boruta | **146 → 23** |

**Çıktı:** `data/processed/prepared_50k.joblib` (model-ready train/test).
**Leakage durumu:** Tüm dönüşümler train'de fit edildi; scaling + SMOTE `03_modeling`'de CV içinde.

**Sonraki adım (03_modeling):** 4 modeli imblearn Pipeline (scaling + SMOTE + model) ile,
RandomizedSearchCV (5-fold, n_iter=100) ile eğit; AUC/Gini/F1, optimal threshold, paired t-test, SHAP.